In [0]:
# COMMAND ----------
# 03 - RISK SCORING & LOGISTIC LAYERS

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

df = spark.table("fraud.scored_xgb_iso").toPandas()

# Final risk score
df["final_risk_score"] = (
    df["fraud_probability"] * 0.6 +
    df["anomaly_flag"] * 0.2 +
    df["duplicate_flag"] * 0.2
)

# Discount capture model
df["date"] = pd.to_datetime(df["date"])
terms_map = {'standard': 30, 'expense': 15, 'low_volume': 45}
df['due_days'] = df['scenario'].map(terms_map).fillna(30)
df['net_due_date'] = df['date'] + pd.to_timedelta(df['due_days'], unit='D')
reference_date = df['date'].max()
df['days_until_discount_expiry'] = (df['net_due_date'] - reference_date).dt.days
df['discount_captured_flag'] = np.where(
    (df['amount'] < 50000) & (df['benford_score'] < 0.1), 1, 0
)

X1 = df[['days_until_discount_expiry', 'invoice_count', 'amount']]
y1 = df['discount_captured_flag']
scaler1 = StandardScaler()
X1_scaled = scaler1.fit_transform(X1)
lr_discount = LogisticRegression(class_weight='balanced')
lr_discount.fit(X1_scaled, y1)
df['prob_capture_discount'] = lr_discount.predict_proba(X1_scaled)[:, 1]

# Late payment risk
df['invoice_amount'] = df['amount']
df['posting_month'] = df['date'].dt.month
df['is_late_flag'] = np.where((df['anomaly_flag'] == 1) & (df['invoice_count'] > 100), 1, 0)
df['vendor_past_late_ratio'] = df.groupby('vendor_id')['is_late_flag'].transform('mean')

X2 = df[['vendor_past_late_ratio', 'invoice_amount', 'posting_month']]
y2 = df['is_late_flag']
scaler2 = StandardScaler()
X2_scaled = scaler2.fit_transform(X2)
late_model = LogisticRegression(class_weight='balanced')
late_model.fit(X2_scaled, y2)
df['late_payment_risk_score'] = late_model.predict_proba(X2_scaled)[:, 1]

# Manual vs automated
df['is_weekend_entry'] = df['is_weekend'].astype(int)
df['is_outside_office_hours'] = np.where(df['anomaly_flag'] == 1, 1, 0)
df['user_manual_entry_history'] = df.groupby('vendor_id')['is_weekend_entry'].transform('mean')
df['is_manual_flag'] = np.where(
    (df['is_weekend_entry'] == 1) | (df['anomaly_score'] < -0.1), 1, 0
)

X3 = df[['is_weekend_entry', 'is_outside_office_hours', 'user_manual_entry_history']]
y3 = df['is_manual_flag']
scaler3 = StandardScaler()
X3_scaled = scaler3.fit_transform(X3)
manual_audit_model = LogisticRegression(class_weight='balanced')
manual_audit_model.fit(X3_scaled, y3)
df['manual_pattern_score'] = manual_audit_model.predict_proba(X3_scaled)[:, 1]

spark.createDataFrame(df).write.mode("overwrite").saveAsTable("fraud.risk_scored_full")
